# CS598JBR MP4 - agentic C-to-Rust translation (Team 12)
Option A: free open-weight model **qwen3:8b** served by Ollama on the Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).
No API key needed. If the runtime disconnects: re-run cells 0-3, then the `--resume` cell.

## 0. Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/picolimpid/CS598JBR-Team-12.git
%cd CS598JBR-Team-12/MP4
!git pull
seed = "191374802308824446074400153398805442526"   # MP3 leftover, not used by MP4
humaneval_input_dataset = "selected_humaneval_" + seed + ".jsonl"

In [ ]:
!nvidia-smi

## 1. Rust toolchain nightly-2025-05-13 (+clippy, rustfmt) and the newmetrics unsafe-metrics tool (~5-10 min)

In [ ]:
import os
!bash scripts/setup_colab.sh
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!cargo +nightly-2025-05-13 --version && cargo +nightly-2025-05-13 clippy --version && which cargo-newmetrics

## 2. Install Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Start the Ollama server in the background and pull qwen3:8b (~5 GB)

In [ ]:
import subprocess, time, urllib.request

LOCAL_MODEL = "qwen3:8b"
CONFIG = "configs/config.ollama.json"

def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not ollama_up():
    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="32768", OLLAMA_KEEP_ALIVE="-1",
               OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
print("ollama up:", ollama_up())
!bash scripts/setup_ollama.sh {LOCAL_MODEL}

## 4. Smoke test: build the crate skeleton + baseline cargo/clippy/newmetrics checks (no LLM)

In [ ]:
!python3 src/pipeline.py --project 1 --config {CONFIG} --dry-run
!python3 src/pipeline.py --project 2 --config {CONFIG} --dry-run

## 5. Run the agent on project_1 (satc). Takes a while; keep the tab open.

In [ ]:
!python3 src/pipeline.py --project 1 --config {CONFIG} --model {LOCAL_MODEL}

## 6. Run the agent on project_2 (jccc)

In [ ]:
!python3 src/pipeline.py --project 2 --config {CONFIG} --model {LOCAL_MODEL}

Only after a disconnect: re-run cells 0-3, then continue the interrupted project with `--resume` (change 1 -> 2 as needed).

In [ ]:
# !python3 src/pipeline.py --project 1 --config {CONFIG} --model {LOCAL_MODEL} --resume

## 7. Final evaluation: build, tests, Clippy per group, newmetrics -> tables for the report

In [ ]:
!python3 src/evaluate.py --config {CONFIG}
!cat trajectories/project_1/summary.md trajectories/project_2/summary.md

## 8. Download the results (translations + trajectories), then commit them to the repo locally

In [ ]:
!zip -qr /content/mp4_results.zip translations trajectories -x '*/target/*'
from google.colab import files
files.download("/content/mp4_results.zip")